# 03 · How well the predictor matches the compiler

Fits the register and shared-memory constants on the training shapes and compares predictions
with what ptxas actually produced, for both the fitted model and the purely analytic one.

Registers and shared memory depend on the config, not on the matrix shape, so these accuracy
numbers are in-sample over configs. The held-out shapes matter for the search results in notebook 04.

In [ ]:
# Setup: mount Drive, make sure the pinned torch/triton are installed, import archtune.
import importlib.metadata as md, os, subprocess, sys
PROJECT = "/content/drive/MyDrive/archtune"
NEED_GPU_STACK = False

if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)

def _ver(pkg):
    try:
        return md.version(pkg).split("+")[0]
    except md.PackageNotFoundError:
        return None

if NEED_GPU_STACK and (_ver("torch"), _ver("triton")) != ("2.6.0", "3.2.0"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.6.0", "triton==3.2.0"], check=True)
    if "torch" in sys.modules:
        raise SystemExit("torch was already imported: restart the session and run this cell again")

from archtune import config, gpu, model, compiler, run
ENV = gpu.capture()
MINI = config.mini_mode()
RESULTS = os.environ.get("ARCHTUNE_RESULTS") or os.path.join(PROJECT, "results_mini" if MINI else "results")
print(ENV["gpu_name"], "| torch", ENV["torch_version"], "| triton", ENV["triton_version"], "|", RESULTS)
if NEED_GPU_STACK and not gpu.pinned_ok(ENV):
    raise SystemExit("wrong torch/triton loaded: restart the session and run this cell again")

In [ ]:
import json, os, pandas as pd
GPU = ENV["gpu_tag"] or "rtx3050_sm86"   # results folder to analyse
summary = run.analyze(RESULTS, GPU, refit=True)
print(json.dumps(summary["models"], indent=1))
print(json.dumps(summary["predictor"], indent=1))
pm = pd.read_parquet(os.path.join(RESULTS, GPU, "analysis", "predictor.parquet"))
run.fig2_predictor(pm, f"Predicted vs measured — {GPU}", os.path.join(RESULTS, "figures", f"fig2_predictor_{GPU}.png"))

## Largest register mispredictions

In [ ]:
top = pd.read_csv(os.path.join(RESULTS, GPU, "analysis", "top_mispredictions.csv"))
sweep_df = run.load_sweep(RESULTS, GPU)
feat_cols = [c for c in sweep_df.columns if c.startswith("ptx_")]
top = top.join(sweep_df.drop_duplicates("cfg_hash").set_index("cfg_hash")[feat_cols], on="cfg_hash")
top[[c for c in top.columns if c.startswith("cfg_")] + ["n_regs", "regs_pred", "n_spills", "regs_abs_err"] + feat_cols]